# 01 — Extraction : jointure des 7 tables IMDb

Objectif : construire **un seul dataset** (`output/imdb_titres.csv`) avec **une ligne par titre IMDb**,
tous types confondus (film, court-métrage, série, épisode, téléfilm, jeu vidéo…).

| Table | Rôle dans la jointure |
|---|---|
| `title.basics` | table de départ : type, titres, années, durée, genres, isAdult |
| `title.ratings` | note moyenne et nombre de votes |
| `title.crew` | réalisateurs et scénaristes |
| `title.principals` | casting et équipe principale, une colonne par métier |
| `name.basics` | remplace les identifiants de personnes (`nm…`) par les noms |
| `title.akas` | titre français, titres alternatifs, pays et langues |
| `title.episode` | série parente, saison et numéro pour les épisodes |

**Principe** : chaque table est d'abord ramenée à *une ligne par titre* (les valeurs multiples sont regroupées
dans une cellule, séparées par ` | `), puis rattachée à `title.basics` par `LEFT JOIN` sur `tconst`.
Aucun titre n'est donc perdu ni dupliqué.

Les fichiers font ~9,6 Go : on utilise **DuckDB**, qui travaille sur disque et ne charge pas tout en mémoire.
Durée d'exécution complète : ~5 minutes.

## 1. Configuration

In [1]:
import time
from pathlib import Path

import duckdb


def trouver_racine():
    # Remonte depuis le dossier courant jusqu'au dossier qui contient Data/
    depart = Path.cwd()
    for dossier in [depart, *depart.parents]:
        if (dossier / "Data").is_dir():
            return dossier
    raise FileNotFoundError(f"Dossier Data/ introuvable au-dessus de {depart}")


RACINE = trouver_racine()
DOSSIER_TSV = RACINE / "Data"
DOSSIER_OUTPUT = RACINE / "output"
DOSSIER_TEMP = DOSSIER_OUTPUT / "temp_duckdb"
SORTIE = DOSSIER_OUTPUT / "imdb_titres.csv"
SEP = " | "      # séparateur à l'intérieur des cellules multi-valeurs
NB_PAQUETS = 16  # les gros regroupements sont faits par paquets de titres pour tenir en mémoire

# IMDb : TSV, pas de guillemets, valeurs manquantes "\N". Tout est lu en texte.
OPTIONS = "delim='\t', quote='', escape='', header=true, nullstr='\\N', all_varchar=true"

print("Fichiers source :")
for f in sorted(DOSSIER_TSV.glob("*.tsv")):
    print(f"  {f.name:<22} {f.stat().st_size / 1e9:5.2f} Go")

Fichiers source :
  name.basics.tsv         0.97 Go
  title.akas.tsv          3.00 Go
  title.basics.tsv        1.11 Go
  title.crew.tsv          0.42 Go
  title.episode.tsv       0.26 Go
  title.principals.tsv    4.55 Go
  title.ratings.tsv       0.03 Go


In [2]:
def table(nom):
    return f"read_csv('{DOSSIER_TSV / (nom + '.tsv')}', {OPTIONS})"


def par_paquets(requete):
    # Une requête par paquet de titres : {paquet} est remplacé par un filtre sur hash(tconst)
    return [requete.replace("{paquet}", f"hash(tconst) % {NB_PAQUETS} = {i}") for i in range(NB_PAQUETS)]


# Base DuckDB temporaire sur disque : les tables intermédiaires n'ont pas à tenir en RAM
DOSSIER_TEMP.mkdir(parents=True, exist_ok=True)
BASE = DOSSIER_TEMP / "jointure_tmp.duckdb"
BASE.unlink(missing_ok=True)
con = duckdb.connect(str(BASE))
con.execute("SET memory_limit = '12GB'")
con.execute("SET preserve_insertion_order = false")
con.execute(f"SET temp_directory = '{DOSSIER_TEMP / 'spill'}'")


def creer(nom, requetes):
    # Crée une table intermédiaire ; une liste de requêtes = table remplie paquet par paquet
    t = time.time()
    if isinstance(requetes, str):
        requetes = [requetes]
    con.execute(f"CREATE OR REPLACE TABLE {nom} AS {requetes[0]}")
    for requete in requetes[1:]:
        con.execute(f"INSERT INTO {nom} {requete}")
    n = con.sql(f"SELECT count(*) FROM {nom}").fetchone()[0]
    print(f"{nom:<15} {n:>12,} lignes  ({time.time() - t:.0f} s)")

## 2. `title.basics` — la table de départ
Une ligne par titre. Les colonnes numériques sont converties en entiers.

In [3]:
creer("titres", f'''
    SELECT tconst, titleType, primaryTitle, originalTitle,
           isAdult::INT AS isAdult, startYear::INT AS startYear, endYear::INT AS endYear,
           runtimeMinutes::INT AS runtimeMinutes, genres
    FROM {table('title.basics')}
''')
con.sql("SELECT titleType, count(*) AS n FROM titres GROUP BY 1 ORDER BY 2 DESC").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

titres            12,786,513 lignes  (4 s)


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,titleType,n
0,tvEpisode,9885938
1,short,1157087
2,movie,756726
3,video,331201
4,tvSeries,305213
5,tvMovie,156109
6,tvMiniSeries,72904
7,tvSpecial,60078
8,videoGame,50193
9,tvShort,11063


## 3. `name.basics` — les personnes
Sert de dictionnaire `nconst → nom` pour les tables crew et principals.

In [4]:
creer("names", f'''
    SELECT nconst, primaryName, birthYear, deathYear, primaryProfession
    FROM {table('name.basics')}
''')

names             15,655,752 lignes  (1 s)


## 4. `title.ratings` — notes et votes
Déjà une ligne par titre.

In [5]:
creer("ratings", f'''
    SELECT tconst, averageRating::DOUBLE AS averageRating, numVotes::INT AS numVotes
    FROM {table('title.ratings')}
''')

ratings            1,709,568 lignes  (0 s)


## 5. `title.crew` — réalisateurs et scénaristes
Les listes d'identifiants (`nm001,nm002`) sont éclatées, remplacées par les noms via `name.basics`,
puis regroupées dans le même ordre. On garde aussi l'année de naissance, de décès et les métiers du 1er réalisateur.

In [6]:
creer("crew_noms", f'''
    WITH crew AS (SELECT * FROM {table('title.crew')}),
    eclate AS (
        SELECT tconst, 'directors' AS role, unnest(string_split(directors, ',')) AS nconst,
               generate_subscripts(string_split(directors, ','), 1) AS pos
        FROM crew WHERE directors IS NOT NULL
        UNION ALL
        SELECT tconst, 'writers', unnest(string_split(writers, ',')),
               generate_subscripts(string_split(writers, ','), 1)
        FROM crew WHERE writers IS NOT NULL
    )
    SELECT e.tconst,
           string_agg(coalesce(n.primaryName, e.nconst), '{SEP}' ORDER BY e.pos)
               FILTER (WHERE role = 'directors') AS directors,
           string_agg(coalesce(n.primaryName, e.nconst), '{SEP}' ORDER BY e.pos)
               FILTER (WHERE role = 'writers') AS writers,
           count(*) FILTER (WHERE role = 'directors') AS numDirectors,
           count(*) FILTER (WHERE role = 'writers') AS numWriters,
           first(n.birthYear ORDER BY e.pos) FILTER (WHERE role = 'directors') AS director1BirthYear,
           first(n.deathYear ORDER BY e.pos) FILTER (WHERE role = 'directors') AS director1DeathYear,
           first(n.primaryProfession ORDER BY e.pos) FILTER (WHERE role = 'directors') AS director1Professions
    FROM eclate e LEFT JOIN names n USING (nconst)
    GROUP BY e.tconst
''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

crew_noms          8,093,643 lignes  (28 s)


## 6. `title.principals` — casting et équipe principale
~100 millions de lignes. On joint d'abord les noms, puis on agrège **par paquets de titres** pour tenir en mémoire.
Une colonne par métier (`actor` et `actress` regroupés dans `actors`), et `characters` alignée sur `actors`.

In [7]:
# Colonne de sortie -> métiers de title.principals
METIERS = {
    "actors": ("actor", "actress"),
    "producers": ("producer",),
    "composers": ("composer",),
    "cinematographers": ("cinematographer",),
    "editors": ("editor",),
    "productionDesigners": ("production_designer",),
    "castingDirectors": ("casting_director",),
    "selfAppearances": ("self",),
    "archiveFootage": ("archive_footage", "archive_sound"),
}
colonnes_metiers = ",\n".join(
    f"string_agg(primaryName, '{SEP}' ORDER BY ordering) "
    f"FILTER (WHERE category IN ({', '.join(repr(c) for c in cats)})) AS {col}"
    for col, cats in METIERS.items()
)

creer("principals_brut", f'''
    SELECT p.tconst, p.ordering::INT AS ordering, p.category,
           coalesce(n.primaryName, p.nconst) AS primaryName,
           -- ["Rick","Narrator"] -> Rick / Narrator
           nullif(replace(regexp_replace(p.characters, '^\\["|"\\]$', '', 'g'),
                          '","', ' / '), '') AS character
    FROM {table('title.principals')} p
    LEFT JOIN names n USING (nconst)
''')

creer("principals_agg", par_paquets(f'''
    SELECT tconst,
           {colonnes_metiers},
           string_agg(coalesce(character, '?'), '{SEP}' ORDER BY ordering)
               FILTER (WHERE category IN ('actor', 'actress')) AS characters,
           count(*) FILTER (WHERE category IN ('actor', 'actress')) AS numActors,
           count(*) FILTER (WHERE category = 'actress') AS numActresses
    FROM principals_brut
    WHERE {{paquet}}
    GROUP BY tconst
'''))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

principals_brut  101,749,592 lignes  (9 s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

principals_agg    11,561,425 lignes  (155 s)


## 7. `title.akas` — titres alternatifs, pays, langues
~60 millions de lignes. On en tire le titre français (région `FR`), le nombre de titres alternatifs,
et les listes distinctes de pays, langues et titres.

In [8]:
creer("akas_brut", f'''
    SELECT titleId AS tconst, ordering::INT AS ordering, title, region, language
    FROM {table('title.akas')}
''')

creer("akas_agg", par_paquets(f'''
    SELECT tconst,
           first(title ORDER BY ordering) FILTER (WHERE region = 'FR') AS frenchTitle,
           count(*) AS numAkas,
           count(DISTINCT region) AS numRegions,
           array_to_string(list_sort(list(DISTINCT region)), '{SEP}') AS regions,
           array_to_string(list_sort(list(DISTINCT language)), '{SEP}') AS languages,
           array_to_string(list_sort(list(DISTINCT title)), '{SEP}') AS akaTitles
    FROM akas_brut
    WHERE {{paquet}}
    GROUP BY tconst
'''))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

akas_brut         59,317,381 lignes  (4 s)


akas_agg          12,782,090 lignes  (23 s)


## 8. `title.episode` — épisodes de séries
Pour chaque épisode : la série parente (identifiant et titre), la saison et le numéro.

In [9]:
creer("episodes", f'''
    SELECT e.tconst, e.parentTconst, s.primaryTitle AS parentTitle,
           e.seasonNumber::INT AS seasonNumber, e.episodeNumber::INT AS episodeNumber
    FROM {table('title.episode')} e
    LEFT JOIN titres s ON s.tconst = e.parentTconst
''')

episodes           9,885,938 lignes  (2 s)


## 9. Jointure finale

In [10]:
creer("resultat", '''
    SELECT f.tconst, f.titleType,
           f.primaryTitle, f.originalTitle, a.frenchTitle,
           f.startYear, f.endYear, f.runtimeMinutes, f.genres, f.isAdult,
           r.averageRating, r.numVotes,
           c.directors, c.numDirectors,
           c.director1BirthYear, c.director1DeathYear, c.director1Professions,
           c.writers, c.numWriters,
           p.actors, p.characters, p.numActors, p.numActresses,
           p.producers, p.composers, p.cinematographers, p.editors,
           p.productionDesigners, p.castingDirectors, p.selfAppearances, p.archiveFootage,
           a.numAkas, a.numRegions, a.regions, a.languages, a.akaTitles,
           e.parentTconst, e.parentTitle, e.seasonNumber, e.episodeNumber
    FROM titres f
    LEFT JOIN ratings        r USING (tconst)
    LEFT JOIN crew_noms      c USING (tconst)
    LEFT JOIN principals_agg p USING (tconst)
    LEFT JOIN akas_agg       a USING (tconst)
    LEFT JOIN episodes       e USING (tconst)
''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

resultat          12,786,513 lignes  (22 s)


## 10. Contrôles
La jointure doit être 1-1 : autant de lignes que de titres dans `title.basics`, sans doublon.

In [11]:
nb_lignes, nb_titres = con.sql("SELECT count(*), count(DISTINCT tconst) FROM resultat").fetchone()
n_base = con.sql("SELECT count(*) FROM titres").fetchone()[0]
assert nb_lignes == nb_titres == n_base, "Une jointure n'est pas 1-1 : titres perdus ou dupliqués"
print(f"OK : {nb_lignes:,} lignes = {n_base:,} titres, aucun doublon")

OK : 12,786,513 lignes = 12,786,513 titres, aucun doublon


In [12]:
# Exemple : Le fabuleux destin d'Amélie Poulain
con.sql("SELECT * FROM resultat WHERE tconst = 'tt0211915'").df().T

,0
tconst,tt0211915
titleType,movie
primaryTitle,Amélie
originalTitle,Le fabuleux destin d'Amélie Poulain
frenchTitle,Amélie
startYear,2001
endYear,<NA>
runtimeMinutes,122
genres,"Comedy,Romance"
isAdult,0


## 11. Export CSV et nettoyage

In [13]:
t = time.time()
con.execute(f"COPY (SELECT * FROM resultat ORDER BY tconst) TO '{SORTIE}' (HEADER, DELIMITER ',')")
con.close()
BASE.unlink()
for f in (DOSSIER_TEMP / "spill").glob("*"):
    f.unlink()
for d in (DOSSIER_TEMP / "spill", DOSSIER_TEMP):
    if d.is_dir() and not any(d.iterdir()):
        d.rmdir()
print(f"{nb_lignes:,} titres -> {SORTIE}  ({SORTIE.stat().st_size / 1e9:.1f} Go, {time.time() - t:.0f} s)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

12,786,513 titres -> /Users/ephrayyy/Documents/CNC/Projet/Data_Extraction/output/imdb_titres.csv  (5.8 Go, 8 s)
